# F5-TTS Dubbing Colab (standalone)
Dub a video (or SRT alone) with **F5-TTS** — clean modern English + voice cloning from a short sample.

This is a **separate** notebook. It does NOT touch your SoniSlowVideo fork or its Chinese Kokoro setup.

**Workflow:**
1. Provide a short clean voice sample (`.wav`/`.mp3`) + the exact text it says → that's your cloned voice.
2. Upload a video + `.srt`, or just a `.srt` (→ mp3).
3. Run → get an mp4 (video + new voice in sync) or mp3.

T4 GPU only.

In [ ]:
# Cell 1 - Install
import subprocess, sys
def run(cmd):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

run('apt-get install -y -qq ffmpeg > /dev/null 2>&1 || true')
run('pip -q install f5-tts pydub')

## 1) Upload your files
Run the cell below, then upload:
- **Video** (optional, for mp4 output): your movie/clip `.mp4`
- **SRT** (required): your subtitle file
- **Voice sample** (required): a short clean clip of the voice you want to clone (English), e.g. 5-20s, no music/noise. Plus the **exact text** it says.

In [ ]:
from google.colab import files
import os

print('Upload: video.mp4 (optional), video.srt (required), voice.wav (required)')
up = files.upload()

work = '/content/f5work'
os.makedirs(work, exist_ok=True)

video_path = None
srt_path = None
voice_path = None
for name, data in up.items():
    with open(os.path.join(work, name), 'wb') as f:
        f.write(data)
    low = name.lower()
    if low.endswith('.srt'):
        srt_path = os.path.join(work, name)
    elif low.endswith(('.mp4','.mkv','.mov','.webm')):
        video_path = os.path.join(work, name)
    elif low.endswith(('.wav','.mp3','.m4a','.flac','.ogg')):
        voice_path = os.path.join(work, name)

print('Video:', video_path)
print('SRT  :', srt_path)
print('Voice:', voice_path)

## 2) Reference text + settings
`REF_TEXT` must be the **exact words spoken** in your voice sample. Keep it to one sentence if possible.

In [ ]:
# Settings
REF_TEXT = "THIS IS THE EXACT TEXT SAID IN YOUR VOICE SAMPLE"   # <-- edit me
SPEED   = 1.0    # 1.0 = natural. 1.1 = slightly faster (your usual preference).
N_FE    = 32     # quality vs speed. 32 = good, 16 = faster.
CFG     = 2.0    # voice strength / stability.

# Sync policy: fit each line into its SRT window.
MIN_SPEED = 0.80   # slowest we'll make a line (lower = more natural but may overflow).
MAX_SPEED = 1.30   # fastest we'll speed a line to keep it inside its window.
GAP_S     = 0.15   # small silence pad after each line.

assert voice_path is not None, 'Upload a voice sample first.'
assert srt_path  is not None, 'Upload an SRT first.'

## 3) Parse the SRT

In [ ]:
import re

def ts_to_sec(t):
    h,m,s = t.replace(',', '.').split(':')
    return int(h)*3600 + int(m)*60 + float(s)

def parse_srt(path):
    text = open(path, encoding='utf-8', errors='replace').read()
    blocks = re.split(r'\n\s*\n', text.strip())
    segs = []
    for b in blocks:
        lines = [l.strip() for l in b.split('\n') if l.strip()]
        if len(lines) < 2:
            continue
        m = re.match(r'(\d{1,2}:\d{2}:\d{2}[.,]\d{3})\s*-->\s*(\d{1,2}:\d{2}:\d{2}[.,]\d{3})', lines[1])
        if not m:
            continue
        txt = ' '.join(lines[2:])
        if not txt:
            continue
        segs.append({'start': ts_to_sec(m.group(1)), 'end': ts_to_sec(m.group(2)), 'text': txt})
    return segs

segs = parse_srt(srt_path)
print(f'Parsed {len(segs)} subtitle lines')
for s in segs[:5]:
    print(f"  {s['start']:.2f}-{s['end']:.2f}: {s['text'][:40]}")

## 4) Generate speech for every line with F5-TTS
Uses your voice sample to clone. `speed` is set by F5's internal pace; we fit timing afterward.

Runs on GPU. Long videos = a few minutes per batch.

In [ ]:
import torch, soundfile as sf, os, glob, shutil
from f5_tts.api import F5TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, '| GPU:', torch.cuda.get_device_name(0) if device=='cuda' else 'CPU (slow!)')

tts = F5TTS(device=device)

# reset segment audio dir
seg_dir = os.path.join(work, 'segs')
shutil.rmtree(seg_dir, ignore_errors=True)
os.makedirs(seg_dir, exist_ok=True)

sr = 24000
for i, s in enumerate(segs):
    wav, wsr, _ = tts.infer(
        ref_file=voice_path,
        ref_text=REF_TEXT,
        gen_text=s['text'],
        speed=SPEED,
        cfg_strength=CFG,
        nfe_step=N_FE,
        target_rms=0.1,
    )
    sf.write(os.path.join(seg_dir, f'{i:05d}.wav'), wav, wsr)
    if i % 10 == 0:
        print(f'{i}/{len(segs)} done')

print(f'Generated {len(segs)} lines -> {seg_dir}')

## 5) Fit each line to its SRT time + assemble full audio track
Speeds up any line that overflows its window (capped at MAX_SPEED) so audio stays in sync — no long empty gaps, no overlap.

In [ ]:
from pydub import AudioSegment
import os

SR = 24000

def seg_duration(path):
    return len(AudioSegment.from_file(path))

def fit(path, slot_ms, out_path):
    a = AudioSegment.from_file(path)
    avail = slot_ms - int(GAP_S*1000)
    if avail < 150:
        avail = 150
    speed = a.duration_seconds / (avail/1000.0)
    speed = max(MIN_SPEED, min(MAX_SPEED, speed))
    if speed > 1.01:
        a = a.speedup(playback_speed=speed, chunk_size=150, crossfade=25)
    elif speed < 0.99:
        a = a._spawn(a.raw_data, overrides={'frame_rate': int(a.frame_rate*speed)}).set_frame_rate(a.frame_rate)
    a.export(out_path, format='wav')
    return a.duration_seconds

# Build full track timeline
track = AudioSegment.silent(duration=int((segs[-1]['end'] + 1.0)*1000), frame_rate=SR)
total = AudioSegment.empty()

for i, s in enumerate(segs):
    raw = os.path.join(seg_dir, f'{i:05d}.wav')
    fitp = os.path.join(seg_dir, f'fit_{i:05d}.wav')
    slot_ms = int((s['end'] - s['start'])*1000)
    dur = fit(raw, slot_ms, fitp)
    a = AudioSegment.from_file(fitp)
    start_ms = int(s['start']*1000)
    track = track.overlay(a, position=start_ms)

track_path = os.path.join(work, 'full_track.wav')
track.export(track_path, format='wav')
print('Full track:', track_path, f'{track.duration_seconds:.1f}s')

## 6) Mux + download
Video provided → **mp4** with new voice. SRT only → **mp3**.

In [ ]:
import subprocess

out = os.path.join(work, 'output.mp4' if video_path else 'output.mp3')

if video_path:
    cmd = f"ffmpeg -y -i {video_path} -i {track_path} -c:v copy -c:a aac -b:a 192k -shortest -map 0:v:0 -map 1:a:0 {out}"
else:
    cmd = f"ffmpeg -y -i {track_path} -c:a libmp3lame -b:a 192k {out}"
subprocess.run(cmd, shell=True, check=True)
print('Done:', out)
from google.colab import files
files.download(out)